# Bandit-based recommendation — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def cosine(a,b):
    a=np.asarray(a,dtype=float); b=np.asarray(b,dtype=float)
    return float(a@b/(np.linalg.norm(a)*np.linalg.norm(b)))
def sigmoid(x): return 1/(1+np.exp(-np.asarray(x)))
def dcg(rels): return sum((2**r-1)/math.log2(i+2) for i,r in enumerate(rels))
def ndcg(rels):
    best=sorted(rels, reverse=True)
    return dcg(rels)/dcg(best) if dcg(best)>0 else 0.0
print('setup ok')

## Balancing exploration and exploitation from live feedback
Bandit recommenders choose items while learning their rewards. The examples compute epsilon-greedy choice probabilities, UCB bonuses, Thompson samples, regret, and a short simulated learning curve.

In [ ]:
# 1) epsilon-greedy mostly exploits but leaves probability for exploration
values=np.array([0.05,0.08,0.04]); eps=.2; probs=np.ones(3)*eps/3; probs[np.argmax(values)]+=1-eps
plt.figure(figsize=(6,3)); plt.bar(['A','B','C'],probs); plt.title('epsilon-greedy action probabilities')
assert abs(probs[1]-0.8666666666666667)<1e-12 and abs(probs.sum()-1)<1e-12

In [ ]:
# 2) UCB adds an uncertainty bonus to under-sampled arms
means=np.array([0.05,0.08,0.04]); pulls=np.array([100,20,5]); t=125; ucb=means+np.sqrt(2*np.log(t)/pulls)
plt.figure(figsize=(6,3)); plt.bar(['A','B','C'],ucb); plt.title('UCB scores')
assert int(np.argmax(ucb))==2 and abs(ucb[2]-1.4297213731251746)<1e-9

In [ ]:
# 3) Thompson sampling draws plausible CTRs from Beta posteriors
clicks=np.array([5,8,1]); non=np.array([95,92,19]); mean=(clicks+1)/(clicks+non+2)
plt.figure(figsize=(6,3)); plt.bar(['A','B','C'],mean); plt.title('Beta posterior means')
assert int(np.argmax(mean))==2 and abs(mean[2]-0.09090909090909091)<1e-12

In [ ]:
# 4) regret accumulates when we choose below the best arm
best=.10; chosen=np.array([.05,.08,.04,.10,.08]); regret=np.cumsum(best-chosen)
plt.figure(figsize=(6,3)); plt.plot(np.arange(1,6),regret,'-o'); plt.xlabel('round'); plt.ylabel('cumulative regret'); plt.title('cost of learning')
assert abs(regret[-1]-0.15)<1e-12

In [ ]:
# 5) tiny UCB simulation learns to pull the better arm more often
rng=np.random.default_rng(0); true=np.array([.04,.08]); pulls=np.ones(2); rewards=np.array([0.,1.]); choices=[]
for t in range(3,203):
    means=rewards/pulls; score=means+np.sqrt(2*np.log(t)/pulls); a=int(np.argmax(score)); choices.append(a); r=rng.random()<true[a]; pulls[a]+=1; rewards[a]+=r
plt.figure(figsize=(6,3)); plt.plot(np.cumsum(np.array(choices)==1)/(np.arange(len(choices))+1)); plt.ylim(0,1); plt.title('fraction choosing better arm')
assert pulls[1]>pulls[0]